In [ ]:
!pip install -q timm torch torchvision scikit-learn pandas pillow tqdm kaggle

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from torch.utils.data import DataLoader, Subset
import numpy as np
import torch
import torch.nn as nn
import timm
import torch.optim as optim
from tqdm import tqdm
from torch.amp import autocast, GradScaler
import os

# Dataset

In [ ]:
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

import kagglehub

path = kagglehub.dataset_download(
    "salviohexia/isic-2019-skin-lesion-images-for-classification"
)

print("Dataset path:", path)

Using Colab cache for faster access to the 'isic-2019-skin-lesion-images-for-classification' dataset.
Dataset path: /kaggle/input/isic-2019-skin-lesion-images-for-classification


## Custom Dataset Class

In [ ]:
from torch.utils.data import Dataset
import os
from PIL import Image


class ISICDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform

        self.class_names = sorted([
            d for d in os.listdir(root)
            if os.path.isdir(os.path.join(root, d))
        ])

        self.class_to_idx = {cls: i for i, cls in enumerate(self.class_names)}

        self.samples = []
        for cls in self.class_names:
            cls_path = os.path.join(root, cls)
            label = self.class_to_idx[cls]

            for img_name in os.listdir(cls_path):
                if img_name.lower().endswith((".jpg", ".png", ".jpeg")):
                    self.samples.append((os.path.join(cls_path, img_name), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = Image.open(path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label

# Data Augmentation & Preprocessing

In [ ]:
import torchvision.transforms as T

IMG_SIZE = 465

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),  # NEW
    T.RandomRotation(15),    # slightly stronger
    T.ColorJitter(0.3, 0.3, 0.3),  # stronger
    T.RandomAffine(degrees=0, translate=(0.05, 0.05)),  # NEW
    T.ToTensor(),
    T.Normalize([0.5] * 3, [0.5] * 3)
])

val_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.5] * 3, [0.5] * 3)
])

In [ ]:
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Dataset Splitting

In [ ]:
DATA_PATH = path
base_dataset = ISICDataset(path, transform=None)
num_classes = len(base_dataset.class_names)

labels = np.array([label for _, label in base_dataset.samples])
indices = np.arange(len(labels))

train_idx, val_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=labels
)

train_dataset = ISICDataset(root=path, transform=train_tf)
val_dataset = ISICDataset(root=path, transform=val_tf)

train_ds = Subset(train_dataset, train_idx)
val_ds = Subset(val_dataset, val_idx)

print(len(train_ds), len(val_ds))

20264 5067


# Class Imbalance Analysis

In [ ]:

train_labels = [base_dataset.samples[i][1] for i in train_idx]
class_counts = np.bincount(train_labels, minlength=num_classes)

print("\n=== Training class distribution ===")
for i, cls in enumerate(base_dataset.class_names):
    print(f"{cls:20s}: {class_counts[i]:6d}  ({class_counts[i] / class_counts.sum() * 100:.2f}%)")
print(f"Imbalance ratio (max/min): {class_counts.max() / class_counts.min():.1f}x")



=== Training class distribution ===
AK                  :    694  (3.42%)
BCC                 :   2658  (13.12%)
BKL                 :   2099  (10.36%)
DF                  :    191  (0.94%)
MEL                 :   3618  (17.85%)
NV                  :  10300  (50.83%)
SCC                 :    502  (2.48%)
VASC                :    202  (1.00%)
Imbalance ratio (max/min): 53.9x


## Handling Class Imbalance

In [ ]:
from torch.utils.data import WeightedRandomSampler


USE_SAMPLER = True
USE_FOCAL_LOSS = True

class_weights_for_sampler = 1.0 / np.maximum(class_counts, 1)
sample_weights = [class_weights_for_sampler[label] for label in train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)



# DataLoaders

In [ ]:
train_loader = DataLoader(
    train_ds,
    batch_size=32,
    sampler=sampler if USE_SAMPLER else None,
    shuffle=False if USE_SAMPLER else True,  # sampler and shuffle are mutually exclusive
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)
